# Gradio Frontend Demo

Prototype frontend for the road-scene weather translation pipeline.

In [2]:
!pip -q install gradio diffusers transformers accelerate pillow opencv-python safetensors

In [3]:
import torch
import gc
import numpy as np
import cv2
import gradio as gr

from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from diffusers import ControlNetModel, StableDiffusionControlNetImg2ImgPipeline

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Device:", device)

# load BLIP-2
blip_dtype = torch.float16 if device == "cuda" else torch.float32

print("Loading BLIP-2...")
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=blip_dtype,
    device_map="auto"
)

# load MiDaS
midas_device = torch.device(device)
midas_model = torch.hub.load("intel-isl/MiDaS", "DPT_Hybrid")
midas_model.to(midas_device).eval()

midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = midas_transforms.dpt_transform

# load ControlNet
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=dtype
)

# load SD and ControlNet
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None
).to(device)

# load IP adapter
pipe.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="models",
    weight_name="ip-adapter_sd15.bin"
)

pipe.set_ip_adapter_scale(0.3)

Device: cuda
Loading BLIP-2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

models/ip-adapter_sd15.bin:   0%|          | 0.00/44.6M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

models/image_encoder/model.safetensors:   0%|          | 0.00/2.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/520 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: h94/IP-Adapter
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
def generate_translation(input_image, reference_image, user_prompt):
    if input_image is None:
        raise gr.Error("Please upload an original road-scene image.")

    if reference_image is None:
        raise gr.Error("Please upload a reference weather image.")

    if user_prompt is None or len(user_prompt.strip()) == 0:
        raise gr.Error("Please enter a transformation prompt before generating.")

    input_image = input_image.convert("RGB")
    reference_image = reference_image.convert("RGB")

    init_image = input_image.resize((512, 512))
    ip_image = reference_image.resize((512, 512))

    # BLIP-2 caption
    inputs = processor(input_image, return_tensors="pt").to(blip_model.device)

    with torch.no_grad():
        ids = blip_model.generate(**inputs, max_length=40)

    caption = processor.decode(ids[0], skip_special_tokens=True)

    if not caption or len(caption.strip()) == 0:
    raise gr.Error("BLIP-2 could not generate a valid caption for the input image.")

    final_prompt = (
        f"Scene description: {caption}. "
        f"User prompt: {user_prompt}. "
        "Preserve the original scene layout, object positions, and perspective."
    )

    negative_prompt = (
        "blurry, low quality, warped perspective, distorted geometry, "
        "distorted cars, distorted road, artifacts"
    )

    # MiDaS depth map
    img_np = np.array(input_image)
    input_batch = transform(img_np).to(midas_device)

    with torch.no_grad():
        prediction = midas_model(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img_np.shape[:2],
            mode="bicubic",
            align_corners=False
        ).squeeze(0).squeeze(0)

    depth = prediction.detach().cpu().numpy()

    depth_vis = cv2.normalize(depth, None, 0, 255, norm_type=cv2.NORM_MINMAX)
    depth_vis = depth_vis.astype(np.uint8)

    depth_image = Image.fromarray(depth_vis).convert("RGB").resize((512, 512))

    generator = torch.Generator(device=device).manual_seed(42)

    output = pipe(
        prompt=final_prompt,
        negative_prompt=negative_prompt,
        image=init_image,
        control_image=depth_image,
        ip_adapter_image=ip_image,
        strength=0.8,
        guidance_scale=7.5,
        controlnet_conditioning_scale=1.0,
        num_inference_steps=30,
        generator=generator
    ).images[0]

    gc.collect()
    torch.cuda.empty_cache()

    return output

In [6]:
css = """
#title {
    text-align: center;
    margin-bottom: 10px;
}

#subtitle {
    text-align: center;
    font-size: 16px;
    margin-bottom: 25px;
}

.gradio-container {
    max-width: 1300px !important;
    margin: auto !important;
}

button {
    font-weight: 600 !important;
    border-radius: 8px !important;
}
"""

In [7]:
with gr.Blocks(title="Generative Diffusion Model-based Translation",
               css=css) as demo:
    gr.Markdown(
        """
        <h1 id="title">Weather Translation</h1>
        <p id="subtitle">
        Upload an image, provide a reference conditional image, and enter a target transformation prompt.
        The models used are BLIP-2, MiDaS, ControlNet, IP-Adapter, and Stable Diffusion.
        </p>
        """
    )

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(type="pil", label="Original Image")
            reference_image = gr.Image(type="pil", label="Weather Reference Image")
            user_prompt = gr.Textbox(
                label="Transformation Prompt",
                value="",
                placeholder="e.g. Convert image to snowy winter with heavy snowfall"
            )
            generate_btn = gr.Button("Generate Translation")

        with gr.Column():
            output_image = gr.Image(type="pil", label="Generated Image")

    generate_btn.click(
        fn=generate_translation,
        inputs=[input_image, reference_image, user_prompt],
        outputs=output_image
    )

demo.launch(share=True)

/tmp/ipykernel_3204/3658730033.py:1: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="Generative Diffusion Model-based Translation",


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://daecf801eea83fea6e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
